# Cell 0: Bayes Hatasi Nedir ve Neden Olcmeyiz?

## Temel Kavram

**Bayes hatasi (Bayes error rate, R*)**: Bir siniflandirma probleminde *mumkun olan en iyi modelin bile* yapacagi *minimum hata*dir. Kaynagi model degil, **verinin kendisindeki belirsizliktir**.

### Sezgi

Iki varyant varsa ve butun feature degerleri birebir ayni ama biri benign biri pathogenic ise — hic bir model bunlari ayiramaz, cunku gordugu girdi ayni. Model ne kadar guclu olursa olsun bu ciftin en az birinde yanilir. Bu hata **indirgenemez** (irreducible).

### Bayes Tavanina Ulasabilecegimiz En Yuksek F1

Modelimiz bu tavana ne kadar yakinsa, "daha fazla ugrasmanin" getirisi o kadar azdır.

## Karar Kurali

| Durum | Anlam | Aksiyon |
|---|---|---|
| Model F1 ≈ Bayes-tavan F1 (fark < ~0.03) | Model zaten tavanda | **Bu panelde dur**, energiyi acik-alani olan panele kaydir |
| Model F1 « Bayes-tavan F1 (fark > ~0.10) | Ciddi acik alan var | FE / yeni model / dis feature denemeye devam et |
| Ara bolge | Sinirli ama gercek marj | Maliyet-fayda tart |

## Bu Notebook'un Yontemi

1. **k-NN tabanlı (Cover-Hart sınırı)**: 1-NN Leave-One-Out hatasından Bayes-error bandı `[R*_low, R*_high]` türet
2. **Label tutarsızlığı**: Aynı feature vektörüne sahip örneklerin karşıt labellere sahip olma oranı
3. **Gower mesafesi**: Karışık tipli + ağır eksik veriye uygun mesafe metriği
4. **Dağılım düzeltmesi**: Bayes tavanını hem doğal hem %80/20 (final-realistic) dağılımda hesapla


In [1]:
# Cell 1: Imports & Config

import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

# Gower mesafe - try/except
try:
    import gower
except ImportError:
    print("UYARI: gower paketi yüklü degil. Yüklemek için: pip install gower")
    gower = None

# Project imports
sys.path.insert(0, '/Users/tefe/teknofest_model/teknofest_model')

from config import SEED, PROJECT_ROOT, TEST_SIZE
from src.columns_real import (
    ALL_FEATURE_COLS, CAT_COLS, AA_COLS, AL_COLS, EK_COLS,
    TARGET_COL, ID_COL, 
    get_constant_cols, get_duplicate_col_pairs, 
    get_missing_mask_col_name, PANEL_INFO
)
from src.metrics import compute_all_metrics, optimize_threshold
from sklearn.metrics import f1_score, precision_score, recall_score, matthews_corrcoef

# PDF report
from fpdf import FPDF
from datetime import datetime

# Notebook path setup
os.chdir('/Users/tefe/teknofest_model/teknofest_model')

# Output directories
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v23_bayes_error_ceiling')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"SEED={SEED}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Gower available: {gower is not None}")


SEED=42
PROJECT_ROOT=/Users/tefe/teknofest_model/teknofest_model
Results dir: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling
Gower available: True


In [2]:
# Cell 2: load_panel(name) - Load and clean

def load_panel(name: str) -> pd.DataFrame:
    """
    Bir paneli yükle ve temizle.
    - Sabit sutunlari (nunique <= 1) drop
    - Birebir ayni sutun ciftlerini drop
    - M3 missing strategy: >%50 eksik sutunlar icin is_missing_* flag + tum sayisala medyan impute
    
    NOT: Train/test split YOK -- bu veri ozelligini olcuyor, tahmin degil.
    """
    fname = PANEL_INFO[name]['file']
    fpath = os.path.join(PROJECT_ROOT, 'data', 'real_data', fname)
    
    df = pd.read_csv(fpath)
    print(f"\n=== {name} ===\nYüklendi: {df.shape}")
    
    # Sabit sutunlar
    const_cols = get_constant_cols(df)
    if const_cols:
        print(f"Sabit sutunlar drop ({len(const_cols)}): {const_cols[:5]}...")
        df = df.drop(columns=const_cols)
    
    # Birebir ayni ciftler
    dup_pairs = get_duplicate_col_pairs(df)
    dup_cols_to_drop = set()
    for c1, c2 in dup_pairs:
        dup_cols_to_drop.add(c2)  # İkinci sütunu kaldır
    if dup_cols_to_drop:
        print(f"Birebir ayni sutun ciftleri ({len(dup_pairs)}) -> {len(dup_cols_to_drop)} sutun drop")
        df = df.drop(columns=list(dup_cols_to_drop))
    
    # M3 missing strategy: is_missing_* flag + medyan impute
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    missing_ratio = df[numeric_cols].isna().mean()
    high_missing_cols = missing_ratio[missing_ratio > 0.50].index.tolist()
    
    for col in high_missing_cols:
        mask_col = get_missing_mask_col_name(col)
        df[mask_col] = df[col].isna().astype(int)
    
    # Medyan impute tum numeric'ler
    for col in numeric_cols:
        if df[col].isna().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
    
    print(f"M3 missing: {len(high_missing_cols)} sutundan is_missing_* flag, medyan impute")
    print(f"After cleaning: {df.shape}")
    
    return df

# Test load
df_master = load_panel('MASTER')


=== MASTER ===
Yüklendi: (2931, 353)
Sabit sutunlar drop (57): ['AL_80', 'AL_101', 'AL_104', 'AL_107', 'AL_110']...
Birebir ayni sutun ciftleri (16) -> 6 sutun drop
M3 missing: 138 sutundan is_missing_* flag, medyan impute
After cleaning: (2931, 428)


In [3]:
# Cell 3: build_gower_features(df) - Prepare Gower matrix

def build_gower_features(df: pd.DataFrame) -> tuple:
    """
    Gower mesafesi icin feature hazirla.
    - Numeric: min-max normalize
    - Categorical (CAT_*, AA_*): string'e cevir
    - is_missing_*: zaten binary
    
    Return:
        X: array (n, d)
        cat_features: boolean mask (d,) - Gower'a hangi sutunlarin kategorik oldugunu soy
    """
    # Feature kolonu belirle (ID ve Label haric)
    feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
    X = df[feature_cols].copy()
    
    # Numeric sutunlari min-max normalize et
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if not col.startswith('is_missing_'):  # is_missing_* zaten binary
            min_val = X[col].min()
            max_val = X[col].max()
            if max_val > min_val:
                X[col] = (X[col] - min_val) / (max_val - min_val)
    
    # Kategorik sutunlari string'e cevir (NaN -> 'MISSING')
    cat_cols_present = [c for c in (CAT_COLS + AA_COLS) if c in X.columns]
    for col in cat_cols_present:
        X[col] = X[col].astype(str).replace('nan', 'MISSING')
    
    # Cat_features mask (gower icin)
    cat_features = np.array([c in cat_cols_present for c in X.columns])
    
    X_array = X.values
    print(f"Gower features ready: {X_array.shape}")
    print(f"  Numeric: {(~cat_features).sum()}, Categorical: {cat_features.sum()}")
    
    return X_array, cat_features, X.columns.tolist()

# Test
X, cat_features, feat_names = build_gower_features(df_master)
print(f"X shape: {X.shape}, cat_features: {cat_features.sum()} / {len(cat_features)}")

Gower features ready: (2931, 426)
  Numeric: 419, Categorical: 7
X shape: (2931, 426), cat_features: 7 / 426


In [4]:
# Cell 4: knn_bayes_bounds(X, y, ks, metric) - k-NN LOO errors & Cover-Hart bounds

def knn_bayes_bounds(X: np.ndarray, y: np.ndarray, ks=(1,3,5,7), 
                     metric='gower', cat_features=None) -> dict:
    """
    k-NN Leave-One-Out hata oranini hesapla, Cover-Hart bandini tir.
    
    Args:
        X: feature matrix (n, d)
        y: labels (n,) - binary 0/1
        ks: k degerler
        metric: 'gower' (default, requires gower package)
        cat_features: boolean mask for gower
    
    Return:
        dict with keys: 'knn_errors' (k -> error_rate), 'R_star_low', 'R_star_high'
    """
    n = len(y)
    
    # Gower mesafe matrisi
    if metric == 'gower':
        if gower is None:
            raise ValueError("gower package required but not installed")
        dist_matrix = gower.gower_matrix(X, cat_features=cat_features)
    else:
        raise ValueError(f"Unknown metric: {metric}")
    
    # Diagonal -> inf (LOO: own sample cannot be neighbor)
    np.fill_diagonal(dist_matrix, np.inf)
    
    knn_errors = {}
    max_k = max(ks)
    
    for k in ks:
        errors = 0
        for i in range(n):
            # k nearest neighbors (excluding self)
            neighbors_idx = np.argsort(dist_matrix[i])[:k]
            neighbor_labels = y[neighbors_idx]
            # Majority vote
            pred = 1 if np.mean(neighbor_labels) > 0.5 else 0
            if pred != y[i]:
                errors += 1
        error_rate = errors / n
        knn_errors[k] = error_rate
    
    # Cover-Hart bounds from k=1
    R1 = knn_errors.get(1, None)
    if R1 is not None:
        # R*_high = R1
        R_star_high = R1
        
        # R*_low = (1 - sqrt(1 - 2*R1)) / 2 (with guard)
        if 1 - 2*R1 < 0:
            print(f"WARNING: R1={R1:.4f} > 0.5 -> discriminant negative, clamp R_star_low=0")
            R_star_low = 0.0
        else:
            sqrt_term = np.sqrt(1 - 2*R1)
            R_star_low = (1 - sqrt_term) / 2
    else:
        R_star_low = R_star_high = None
    
    return {
        'knn_errors': knn_errors,
        'R_star_low': R_star_low,
        'R_star_high': R_star_high,
        'dist_matrix': dist_matrix
    }

# Test (MASTER)
y = df_master[TARGET_COL].values
print(f"\nTesting k-NN bounds on MASTER (n={len(y)}, labels {np.unique(y)})...")
results_knn = knn_bayes_bounds(X, y, ks=(1,3,5,7), metric='gower', cat_features=cat_features)
print(f"k-NN errors: {results_knn['knn_errors']}")
print(f"R* bounds: [{results_knn['R_star_low']:.4f}, {results_knn['R_star_high']:.4f}]")


Testing k-NN bounds on MASTER (n=2931, labels [0 1])...
k-NN errors: {1: 0.28863868986693964, 3: 0.2613442511088366, 5: 0.24530876833845103, 7: 0.2374616171954964}
R* bounds: [0.1749, 0.2886]


In [5]:
# Cell 5: label_inconsistency(X, y, dist, eps_percentiles) - Feature-space label conflict

def label_inconsistency(X: np.ndarray, y: np.ndarray, dist_matrix: np.ndarray,
                        eps_percentiles=(5, 10, 25)) -> dict:
    """
    Aynı (veya çok yakın) feature vektörüne sahip örnekleri grupla,
    grup-içi label çelişkisini ölç.
    
    Return:
        dict with 'exact_collision' (%) ve 'eps_neighborhood' ({eps: ratio})
    """
    n = len(y)
    
    # Exact collision: birebir ayni feature (mesafe = 0)
    exact_collision_count = 0
    exact_collision_conflicts = 0
    
    # Strict: 0 dist (float precision icinde)
    for i in range(n):
        for j in range(i+1, n):
            if dist_matrix[i, j] < 1e-10:  # Effectively 0
                exact_collision_count += 1
                if y[i] != y[j]:
                    exact_collision_conflicts += 1
    
    exact_collision_ratio = exact_collision_conflicts / max(1, exact_collision_count) if exact_collision_count > 0 else 0
    
    # Epsilon-komşuluk tutarsızlığı
    eps_inconsistency = {}
    for percentile in eps_percentiles:
        # Mesafe dagiliminin percentile'ini eps olarak kullan
        dist_nondiag = dist_matrix[np.triu_indices_from(dist_matrix, k=1)]
        dist_nondiag = dist_nondiag[dist_nondiag < np.inf]
        eps = np.percentile(dist_nondiag, percentile) if len(dist_nondiag) > 0 else 0
        
        conflict_count = 0
        total_pairs = 0
        for i in range(n):
            for j in range(i+1, n):
                if dist_matrix[i, j] <= eps:
                    total_pairs += 1
                    if y[i] != y[j]:
                        conflict_count += 1
        
        conflict_ratio = conflict_count / max(1, total_pairs) if total_pairs > 0 else 0
        eps_inconsistency[f"eps_p{percentile}"] = {"eps": eps, "conflict_ratio": conflict_ratio}
    
    return {
        "exact_collision_pairs": exact_collision_count,
        "exact_collision_conflict_ratio": exact_collision_ratio,
        "eps_neighborhoods": eps_inconsistency
    }

# Test
print("\nTesting label inconsistency on MASTER...")
results_incon = label_inconsistency(X, y, results_knn["dist_matrix"], eps_percentiles=(5,10,25))
print(f"Exact collision pairs: {results_incon['exact_collision_pairs']}")
print(f"Exact collision conflict ratio: {results_incon['exact_collision_conflict_ratio']:.4f}")
print(f"Eps-neighborhood conflicts: {results_incon['eps_neighborhoods']}")


Testing label inconsistency on MASTER...
Exact collision pairs: 6670
Exact collision conflict ratio: 0.4610
Eps-neighborhood conflicts: {'eps_p5': {'eps': 0.012210583779960872, 'conflict_ratio': 0.258043931885084}, 'eps_p10': {'eps': 0.027441303804516794, 'conflict_ratio': 0.24829060625256177}, 'eps_p25': {'eps': 0.0833258256316185, 'conflict_ratio': 0.3079575846383581}}


In [6]:
# Cell 6: bayes_error_to_f1(X, y, dist, target_benign_frac, n_boot) - Bootstrap Bayes-optimal F1

def bayes_error_to_f1(X: np.ndarray, y: np.ndarray, dist_matrix: np.ndarray,
                      target_benign_frac=None, n_boot=50, seed=SEED) -> dict:
    """
    Verilen hedef dağılımda Bayes-optimal tahminin F1'ini hesapla.
    
    Args:
        target_benign_frac: None (doğal dağılım) veya 0.80 (final %80/20)
        n_boot: bootstrap örnek sayısı
    
    Return:
        dict: {'f1_mean', 'f1_std', 'f1_ci_low', 'f1_ci_high', 'precision_mean', ...}
    """
    n = len(y)
    np.random.seed(seed)
    
    # Doğal dağılım
    n_benign = (y == 0).sum()
    benign_ratio = n_benign / n
    
    if target_benign_frac is None:
        target_benign_frac = benign_ratio
    
    f1_scores = []
    precision_scores = []
    recall_scores = []
    mcc_scores = []
    
    for _ in range(n_boot):
        # Target dağılıma uyan bootstrap havuzu oluştur
        benign_idx = np.where(y == 0)[0]
        patho_idx = np.where(y == 1)[0]
        
        n_target_benign = int(len(benign_idx) * target_benign_frac / benign_ratio)
        n_target_patho = int(len(patho_idx) * (1 - target_benign_frac) / (1 - benign_ratio))
        
        sampled_benign = np.random.choice(benign_idx, min(n_target_benign, len(benign_idx)), replace=False)
        sampled_patho = np.random.choice(patho_idx, min(n_target_patho, len(patho_idx)), replace=False)
        
        boot_idx = np.concatenate([sampled_benign, sampled_patho])
        
        # Bayes-optimal sınıflandırıcı: k-NN (k=5) majority vote
        y_boot = y[boot_idx]
        y_pred = np.zeros_like(y_boot)
        
        for ii, i in enumerate(boot_idx):
            neighbors_idx = np.argsort(dist_matrix[i])[:5]
            neighbor_labels = y[neighbors_idx]
            y_pred[ii] = 1 if np.mean(neighbor_labels) > 0.5 else 0
        
        # Pathogenic (label=1) metrikler
        f1_scores.append(f1_score(y_boot, y_pred, zero_division=0))
        precision_scores.append(precision_score(y_boot, y_pred, zero_division=0))
        recall_scores.append(recall_score(y_boot, y_pred, zero_division=0))
        mcc_scores.append(matthews_corrcoef(y_boot, y_pred))
    
    f1_arr = np.array(f1_scores)
    f1_mean = np.mean(f1_arr)
    f1_std = np.std(f1_arr)
    f1_ci = np.percentile(f1_arr, [2.5, 97.5])
    
    return {
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "f1_ci_low": f1_ci[0],
        "f1_ci_high": f1_ci[1],
        "precision_mean": np.mean(precision_scores),
        "recall_mean": np.mean(recall_scores),
        "mcc_mean": np.mean(mcc_scores)
    }

# Test (MASTER, natural + 80/20)
print("\nTesting Bayes-optimal F1 on MASTER...")
bayes_natural = bayes_error_to_f1(X, y, results_knn["dist_matrix"], target_benign_frac=None, n_boot=10)
print(f"Bayes-F1 (natural): {bayes_natural['f1_mean']:.4f} +/- {bayes_natural['f1_std']:.4f}")

bayes_8020 = bayes_error_to_f1(X, y, results_knn["dist_matrix"], target_benign_frac=0.80, n_boot=10)
print(f"Bayes-F1 (80/20): {bayes_8020['f1_mean']:.4f} +/- {bayes_8020['f1_std']:.4f}")



Testing Bayes-optimal F1 on MASTER...
Bayes-F1 (natural): 0.8426 +/- 0.0000
Bayes-F1 (80/20): 0.6534 +/- 0.0055


In [7]:
# Cell 7: run_panel(name) - Run all analyses for one panel

def run_panel(name: str) -> dict:
    """
    Bir panel icin tum Bayes-error analizlerini calistir.
    
    Return:
        dict: tum sonuclari ic ice
    """
    print(f"\n{'='*60}")
    print(f"Running panel: {name}")
    print(f"{'='*60}")
    
    # Load & clean
    df = load_panel(name)
    y = df[TARGET_COL].values
    
    # Build Gower features
    X, cat_features, feat_names = build_gower_features(df)
    
    # k-NN bounds
    print("\nComputing k-NN bounds...")
    knn_result = knn_bayes_bounds(X, y, ks=(1,3,5,7), metric='gower', cat_features=cat_features)
    
    # Label inconsistency
    print("\nComputing label inconsistency...")
    incon_result = label_inconsistency(X, y, knn_result["dist_matrix"], eps_percentiles=(5,10,25))
    
    # Bayes-F1 natural
    print("\nComputing Bayes-F1 (natural distribution)...")
    bayes_nat = bayes_error_to_f1(X, y, knn_result["dist_matrix"], target_benign_frac=None, n_boot=50, seed=SEED)
    
    # Bayes-F1 80/20
    print("\nComputing Bayes-F1 (80/20 final distribution)...")
    bayes_8020 = bayes_error_to_f1(X, y, knn_result["dist_matrix"], target_benign_frac=0.80, n_boot=50, seed=SEED)
    
    # Collect results
    n_samples = len(y)
    n_benign = (y == 0).sum()
    n_patho = (y == 1).sum()
    
    result = {
        "panel": name,
        "n_samples": n_samples,
        "n_benign": n_benign,
        "n_patho": n_patho,
        "knn_errors": knn_result["knn_errors"],
        "R_star_low": knn_result["R_star_low"],
        "R_star_high": knn_result["R_star_high"],
        "exact_collision_pairs": incon_result["exact_collision_pairs"],
        "exact_collision_conflict_ratio": incon_result["exact_collision_conflict_ratio"],
        "eps_neighborhoods": incon_result["eps_neighborhoods"],
        "bayes_f1_natural": bayes_nat["f1_mean"],
        "bayes_f1_natural_ci": (bayes_nat["f1_ci_low"], bayes_nat["f1_ci_high"]),
        "bayes_f1_8020": bayes_8020["f1_mean"],
        "bayes_f1_8020_ci": (bayes_8020["f1_ci_low"], bayes_8020["f1_ci_high"]),
        "bayes_precision_8020": bayes_8020["precision_mean"],
        "bayes_recall_8020": bayes_8020["recall_mean"],
        "bayes_mcc_8020": bayes_8020["mcc_mean"],
    }
    
    # CFTR low-confidence flag
    if name == 'CFTR':
        result["low_confidence"] = True
        print(f"\n*** CFTR LOW CONFIDENCE FLAG: n_benign={n_benign} (cok azdir)")
    else:
        result["low_confidence"] = False
    
    return result

# Run all panels
panel_results = {}
for panel_name in ['MASTER', 'KANSER', 'PAH', 'CFTR']:
    panel_results[panel_name] = run_panel(panel_name)

print("\n" + "="*60)
print("All panels completed.")
print("="*60)


Running panel: MASTER

=== MASTER ===
Yüklendi: (2931, 353)
Sabit sutunlar drop (57): ['AL_80', 'AL_101', 'AL_104', 'AL_107', 'AL_110']...
Birebir ayni sutun ciftleri (16) -> 6 sutun drop
M3 missing: 138 sutundan is_missing_* flag, medyan impute
After cleaning: (2931, 428)
Gower features ready: (2931, 426)
  Numeric: 419, Categorical: 7

Computing k-NN bounds...

Computing label inconsistency...

Computing Bayes-F1 (natural distribution)...

Computing Bayes-F1 (80/20 final distribution)...

Running panel: KANSER

=== KANSER ===
Yüklendi: (388, 353)
Sabit sutunlar drop (69): ['AL_44', 'AL_47', 'AL_80', 'AL_101', 'AL_104']...
Birebir ayni sutun ciftleri (3) -> 2 sutun drop
M3 missing: 190 sutundan is_missing_* flag, medyan impute
After cleaning: (388, 472)
Gower features ready: (388, 470)
  Numeric: 465, Categorical: 5

Computing k-NN bounds...

Computing label inconsistency...

Computing Bayes-F1 (natural distribution)...

Computing Bayes-F1 (80/20 final distribution)...

Running panel

In [8]:
# Cell 8: Results summary table

# Current best model F1 scores (from NB39 results)
BEST_MODEL_F1 = {
    'MASTER': 0.6379,
    'KANSER': 0.7300,
    'PAH': 0.5820,
    'CFTR': 0.8630
}

# Build summary table
summary_rows = []
for name, res in panel_results.items():
    knn_err_1 = res['knn_errors'].get(1, None)
    knn_err_3 = res['knn_errors'].get(3, None)
    knn_err_5 = res['knn_errors'].get(5, None)
    knn_err_7 = res['knn_errors'].get(7, None)
    
    bayes_f1_8020 = res['bayes_f1_8020']
    bayes_f1_8020_ci = res['bayes_f1_8020_ci']
    
    model_f1 = BEST_MODEL_F1[name]
    gap = bayes_f1_8020 - model_f1
    
    summary_rows.append({
        'Panel': name,
        'n': res['n_samples'],
        'Benign': res['n_benign'],
        'Patho': res['n_patho'],
        'kNN-1': f"{knn_err_1:.4f}" if knn_err_1 else "N/A",
        'kNN-3': f"{knn_err_3:.4f}" if knn_err_3 else "N/A",
        'kNN-5': f"{knn_err_5:.4f}" if knn_err_5 else "N/A",
        'kNN-7': f"{knn_err_7:.4f}" if knn_err_7 else "N/A",
        'R*_low': f"{res['R_star_low']:.4f}" if res['R_star_low'] else "N/A",
        'R*_high': f"{res['R_star_high']:.4f}" if res['R_star_high'] else "N/A",
        'Bayes-F1(8020)': f"{bayes_f1_8020:.4f}",
        'CI': f"[{bayes_f1_8020_ci[0]:.4f}, {bayes_f1_8020_ci[1]:.4f}]",
        'Model-F1': f"{model_f1:.4f}",
        'Gap': f"{gap:.4f}",
        'Low-Conf': '**' if res['low_confidence'] else ''
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*140)
print("BAYES ERROR CEILING - ALL PANELS")
print("="*140)
print(summary_df.to_string(index=False))
print("\nNOTE: ** = Low-confidence (CFTR n_benign=21)")
print("\nKey columns:")
print("  - Bayes-F1(8020): Bayes-optimal F1 on final %80/20 distribution")
print("  - Model-F1: Best model F1 from NB39")
print("  - Gap = Bayes-F1(8020) - Model-F1")

# Save to CSV
summary_df.to_csv(os.path.join(RESULTS_DIR, 'bayes_error_summary.csv'), index=False)
print(f"\nSaved: {os.path.join(RESULTS_DIR, 'bayes_error_summary.csv')}")


BAYES ERROR CEILING - ALL PANELS
 Panel    n  Benign  Patho  kNN-1  kNN-3  kNN-5  kNN-7 R*_low R*_high Bayes-F1(8020)               CI Model-F1     Gap Low-Conf
MASTER 2931     782   2149 0.2886 0.2613 0.2453 0.2375 0.1749  0.2886         0.6540 [0.6449, 0.6658]   0.6379  0.0161         
KANSER  388     120    268 0.1985 0.1778 0.1907 0.1753 0.1117  0.1985         0.7263 [0.6984, 0.7576]   0.7300 -0.0037         
   PAH  372      62    310 0.2177 0.1909 0.1909 0.1747 0.1243  0.2177         0.6945 [0.6779, 0.7081]   0.5820  0.1125         
  CFTR  111      21     90 0.1892 0.1622 0.1441 0.1532 0.1058  0.1892         0.7550 [0.7273, 0.7719]   0.8630 -0.1080       **

NOTE: ** = Low-confidence (CFTR n_benign=21)

Key columns:
  - Bayes-F1(8020): Bayes-optimal F1 on final %80/20 distribution
  - Model-F1: Best model F1 from NB39
  - Gap = Bayes-F1(8020) - Model-F1

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/bayes_error_summary.csv


In [9]:
# Cell 9: Decision table - Apply gap rules

decision_rows = []
for name, res in panel_results.items():
    bayes_f1_8020 = res['bayes_f1_8020']
    model_f1 = BEST_MODEL_F1[name]
    gap = bayes_f1_8020 - model_f1
    
    # Decision rule
    if gap < 0.03:
        decision = "DUR (at ceiling)"
        confidence = "HIGH" if not res['low_confidence'] else "LOW"
    elif gap > 0.10:
        decision = "DEVAM (significant margin)"
        confidence = "HIGH" if not res['low_confidence'] else "LOW"
    else:
        decision = "ARA (marginal)"
        confidence = "HIGH" if not res['low_confidence'] else "LOW"
    
    if res['low_confidence']:
        confidence_note = "n_benign=21, genis CI"
    else:
        confidence_note = ""
    
    decision_rows.append({
        'Panel': name,
        'Gap': f"{gap:.4f}",
        'Decision': decision,
        'Confidence': confidence,
        'Note': confidence_note
    })

decision_df = pd.DataFrame(decision_rows)
print("\n" + "="*100)
print("DECISION TABLE - Which panels to continue optimizing?")
print("="*100)
print(decision_df.to_string(index=False))
print("\nDecision rule:")
print("  Gap < 0.03  --> DUR (panel at ceiling, focus on others)")
print("  Gap > 0.10  --> DEVAM (significant margin left, continue FE/tuning)")
print("  0.03 <= Gap <= 0.10 --> ARA (evaluate cost-benefit)")

decision_df.to_csv(os.path.join(RESULTS_DIR, 'decision_table.csv'), index=False)
print(f"\nSaved: {os.path.join(RESULTS_DIR, 'decision_table.csv')}")


DECISION TABLE - Which panels to continue optimizing?
 Panel     Gap                   Decision Confidence                  Note
MASTER  0.0161           DUR (at ceiling)       HIGH                      
KANSER -0.0037           DUR (at ceiling)       HIGH                      
   PAH  0.1125 DEVAM (significant margin)       HIGH                      
  CFTR -0.1080           DUR (at ceiling)        LOW n_benign=21, genis CI

Decision rule:
  Gap < 0.03  --> DUR (panel at ceiling, focus on others)
  Gap > 0.10  --> DEVAM (significant margin left, continue FE/tuning)
  0.03 <= Gap <= 0.10 --> ARA (evaluate cost-benefit)

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/decision_table.csv


In [10]:
# Cell 10: Visualizations (5 figures)

plt.style.use('seaborn-v0_8-darkgrid')
fig_paths = []

# Figure 1: Model F1 vs Bayes-F1 (80/20) - gap visualization
fig, ax = plt.subplots(figsize=(10, 6))
panels = list(panel_results.keys())
model_f1s = [BEST_MODEL_F1[p] for p in panels]
bayes_f1s = [panel_results[p]['bayes_f1_8020'] for p in panels]
gaps = [b - m for b, m in zip(bayes_f1s, model_f1s)]

x = np.arange(len(panels))
width = 0.35
bars1 = ax.bar(x - width/2, model_f1s, width, label='Best Model F1', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, bayes_f1s, width, label='Bayes-F1 (80/20)', color='darkorange', alpha=0.8)

ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('Model F1 vs Bayes-Error Ceiling (Final %80/20 Distribution)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(panels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add gap labels on top
for i, (gap, bayes_f1) in enumerate(zip(gaps, bayes_f1s)):
    ax.text(i, bayes_f1 + 0.02, f"gap={gap:.3f}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig1_model_vs_ceiling.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
print(f"Saved: {fig_path}")
plt.close()

# Figure 2: k-NN error curve (4 panels)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(panel_results.items()):
    ax = axes[idx]
    ks = sorted(res['knn_errors'].keys())
    errors = [res['knn_errors'][k] for k in ks]
    
    ax.plot(ks, errors, marker='o', linewidth=2, markersize=8, color='steelblue')
    ax.set_xlabel('k (neighbors)', fontsize=11, fontweight='bold')
    ax.set_ylabel('LOO Error Rate', fontsize=11, fontweight='bold')
    ax.set_title(f"{name} (n={res['n_samples']})", fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Add R* band
    if res['R_star_low'] is not None:
        ax.axhline(res['R_star_low'], color='green', linestyle='--', linewidth=1.5, label=f"R*_low={res['R_star_low']:.3f}")
        ax.axhline(res['R_star_high'], color='red', linestyle='--', linewidth=1.5, label=f"R*_high={res['R_star_high']:.3f}")
        ax.legend(fontsize=9)

plt.suptitle('k-NN Leave-One-Out Error Rates & Cover-Hart Bounds', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig2_knn_curve.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
print(f"Saved: {fig_path}")
plt.close()

# Figure 3: Epsilon-neighborhood inconsistency
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(panel_results))
width = 0.25
panel_names = list(panel_results.keys())

for eps_idx, eps_key in enumerate(['eps_p5', 'eps_p10', 'eps_p25']):
    conflict_ratios = []
    for name in panel_names:
        res = panel_results[name]
        ratio = res['eps_neighborhoods'][eps_key]['conflict_ratio']
        conflict_ratios.append(ratio)
    
    ax.bar(x + eps_idx*width, conflict_ratios, width, label=f"{eps_key}", alpha=0.8)

ax.set_ylabel('Label Conflict Ratio', fontsize=12, fontweight='bold')
ax.set_title('Epsilon-Neighborhood Label Inconsistency', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(panel_names, fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig3_eps_inconsistency.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
print(f"Saved: {fig_path}")
plt.close()

# Figure 4: Natural vs 80/20 distribution comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(panel_names))
width = 0.35
nat_f1s = [panel_results[p]['bayes_f1_natural'] for p in panel_names]
f8020_f1s = [panel_results[p]['bayes_f1_8020'] for p in panel_names]

bars1 = ax.bar(x - width/2, nat_f1s, width, label='Natural Distribution', color='skyblue', alpha=0.8)
bars2 = ax.bar(x + width/2, f8020_f1s, width, label='Final %80/%20', color='salmon', alpha=0.8)

ax.set_ylabel('Bayes-Optimal F1', fontsize=12, fontweight='bold')
ax.set_title('Bayes-F1: Natural vs Final %80/20 Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(panel_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig4_dist_comparison.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
print(f"Saved: {fig_path}")
plt.close()

# Figure 5: Decision zones by gap
fig, ax = plt.subplots(figsize=(10, 6))

gaps_vals = []
colors_list = []
for name in panel_names:
    gap = panel_results[name]['bayes_f1_8020'] - BEST_MODEL_F1[name]
    gaps_vals.append(gap)
    if gap < 0.03:
        colors_list.append('red')  # DUR
    elif gap > 0.10:
        colors_list.append('green')  # DEVAM
    else:
        colors_list.append('orange')  # ARA

bars = ax.barh(panel_names, gaps_vals, color=colors_list, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add decision zones
ax.axvline(0.03, color='gray', linestyle='--', linewidth=2, label='DUR threshold (gap=0.03)')
ax.axvline(0.10, color='gray', linestyle='--', linewidth=2, label='DEVAM threshold (gap=0.10)')
ax.axvspan(0, 0.03, alpha=0.1, color='red')
ax.axvspan(0.03, 0.10, alpha=0.1, color='orange')
ax.axvspan(0.10, max(gaps_vals)*1.1, alpha=0.1, color='green')

ax.set_xlabel('Gap = Bayes-F1(80/20) - Model-F1', fontsize=12, fontweight='bold')
ax.set_title('Decision Zones: Should We Continue Optimizing?', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='x', alpha=0.3)

# Add gap values on bars
for i, (panel, gap) in enumerate(zip(panel_names, gaps_vals)):
    ax.text(gap + 0.005, i, f"{gap:.3f}", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig5_decision_zones.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
print(f"Saved: {fig_path}")
plt.close()

print(f"\nAll {len(fig_paths)} figures saved to {RESULTS_DIR}")

Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/fig1_model_vs_ceiling.png
Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/fig2_knn_curve.png
Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/fig3_eps_inconsistency.png
Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/fig4_dist_comparison.png
Saved: /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling/fig5_decision_zones.png

All 5 figures saved to /Users/tefe/teknofest_model/teknofest_model/results/v23_bayes_error_ceiling


In [12]:
# Cell 11: PDF Report

class NB40Report(FPDF):
    """Bayes Error Ceiling report."""
    
    def header(self):
        self.set_font('Helvetica', 'B', 16)
        self.cell(0, 10, 'NB40: Bayes Error Ceiling - Tum Paneller', 0, 1, 'C')
        self.set_font('Helvetica', '', 10)
        self.cell(0, 5, f'SEED={SEED} | k-NN + Label Tutarsizligi | {datetime.now().strftime("%Y-%m-%d")}', 0, 1, 'C')
        self.ln(5)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', '', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')
    
    def section_title(self, title):
        self.set_font('Helvetica', 'B', 13)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)
    
    def body_text(self, text):
        self.set_font('Helvetica', '', 10)
        self.multi_cell(0, 5, text)
        self.ln(2)
    
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 8)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 7)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(2)

# Create report
pdf = NB40Report('P', 'mm', 'A4')
pdf.add_page()

# Section 1: Concept
pdf.section_title('1. Bayes Hatasi Nedir?')
pdf.body_text(
    'Bayes hatasi (R*) bir siniflandirma probleminde mumkun olan en iyi modelin bile yapacagi '
    'minimum hatadir. Kaynagi model degil, verinin kendisindeki belirsizliktir. '
    'Ornek: iki varyant ayni feature degerleri tasiyor ama biri benign biri pathogenic ise, '
    'hic bir model bunlari ayiramaz. Bu hata indirgenemez (irreducible).'
)

pdf.body_text(
    'Bu notebook, k-NN Cover-Hart sinirlari + label tutarsizligi ölcutleri kullanarak '
    'her panelde Bayes hatasi bandini (R*_low, R*_high) tahmin eder, '
    've Bayes-optimal F1 tavanini son test dagilimlarina (final %80/20) gore hesaplar.'
)

# Section 2: Results summary
pdf.section_title('2. Sonuc Ozeti (Tum Paneller)')

# Table: summary results
summary_table_rows = []
for name in panel_names:
    res = panel_results[name]
    knn_1 = res['knn_errors'].get(1, 0)
    r_low = res['R_star_low'] or 0
    r_high = res['R_star_high'] or 0
    bayes_f1 = res['bayes_f1_8020']
    model_f1 = BEST_MODEL_F1[name]
    gap = bayes_f1 - model_f1
    summary_table_rows.append([
        name,
        str(res['n_samples']),
        f"{knn_1:.3f}",
        f"{r_low:.3f}",
        f"{r_high:.3f}",
        f"{bayes_f1:.3f}",
        f"{model_f1:.3f}",
        f"{gap:.3f}"
    ])

pdf.add_table(
    ['Panel', 'n', 'kNN-1', 'R*_low', 'R*_high', 'Bayes-F1', 'Model-F1', 'Gap'],
    summary_table_rows,
    col_widths=[20, 15, 15, 18, 18, 22, 20, 15]
)

# Section 3: Decision table
pdf.section_title('3. Karar Tablosu')

decision_table_rows = []
for name in panel_names:
    gap = panel_results[name]['bayes_f1_8020'] - BEST_MODEL_F1[name]
    if gap < 0.03:
        decision = 'DUR'
    elif gap > 0.10:
        decision = 'DEVAM'
    else:
        decision = 'ARA'
    decision_table_rows.append([name, f"{gap:.3f}", decision])

pdf.add_table(['Panel', 'Gap', 'Karar'], decision_table_rows, col_widths=[40, 40, 60])

pdf.body_text(
    'Karar Kurali: Gap < 0.03 = DUR | 0.03-0.10 = ARA | Gap > 0.10 = DEVAM'
)

# Section 4: Key findings
pdf.section_title('4. Temel Bulgular')

for name in sorted(panel_names):
    res = panel_results[name]
    bayes_f1 = res['bayes_f1_8020']
    model_f1 = BEST_MODEL_F1[name]
    gap = bayes_f1 - model_f1
    cf_mark = ' (low conf)' if res['low_confidence'] else ''
    pdf.body_text(f"{name}: F1={bayes_f1:.3f}, gap={gap:.3f}{cf_mark}")

# Page 2: Visualizations
pdf.add_page()
pdf.section_title('5. Gorsellestirilmis Sonuclar')

for i, fig_path in enumerate(fig_paths, 1):
    pdf.cell(0, 5, f'Sekil {i}: {os.path.basename(fig_path)}', 0, 1)
    try:
        pdf.image(fig_path, x=10, w=190)
    except:
        pdf.body_text(f'[Sekil {i}]')
    pdf.ln(1)

# Output
report_path = os.path.join(PROJECT_ROOT, 'reports', 'NB40_bayes_error_ceiling_report.pdf')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
pdf.output(report_path)
print(f"\nPDF Report saved: {report_path}")


PDF Report saved: /Users/tefe/teknofest_model/teknofest_model/reports/NB40_bayes_error_ceiling_report.pdf
